<a href="https://colab.research.google.com/github/aliraza-chaudhary/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliraza-chaudhary/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [208]:
!git clone https://github.com/aliraza-chaudhary/flyrank-ml-internship.git /content/flyrank-ml-internship

fatal: destination path '/content/flyrank-ml-internship' already exists and is not an empty directory.


In [209]:
import os

print(os.path.exists("/content/flyrank-ml-internship"))
print(os.listdir("/content/flyrank-ml-internship"))

True
['data', 'requirements.txt', 'outputs', 'LICENSE', 'docs', 'skills', '.gitignore', 'work', 'submission', 'DATA_USE.md', 'notebooks', 'AGENTS.md', 'SETUP.md', 'scripts', '.github', 'GUIDE.md', '.git', 'CLAUDE.md', 'README.md']


In [210]:
%cd /content/flyrank-ml-internship

!python scripts/02_baseline_score.py

/content/flyrank-ml-internship
Wrote baseline queue: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340


In [211]:
import os

baseline_path = "/content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv"

print("Baseline exists:", os.path.exists(baseline_path))
print("Path:", baseline_path)

Baseline exists: True
Path: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv


In [212]:
baseline_df = pd.read_csv(baseline_path)

print("Shape:", baseline_df.shape)
print("\nColumns:")
print(baseline_df.columns.tolist())

display(baseline_df.head())

Shape: (30000, 22)

Columns:
['content_id', 'client_id', 'baseline_rank', 'baseline_refresh_score', 'visibility_score', 'freshness_risk_score', 'position_opportunity_score', 'depth_gap_score', 'reason_codes', 'suggested_action_baseline', 'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction']


,content_id,client_id,baseline_rank,baseline_refresh_score,visibility_score,freshness_risk_score,position_opportunity_score,depth_gap_score,reason_codes,suggested_action_baseline,...,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count,trend_direction
0,content_9532f197bbc8,client_4e07408562,1,0.941189,0.999633,0.8432,0.979233,0.871347,declining_with_demand|page_one_decay_risk|low_...,refresh,...,2689,1098,2.0,0.87,8.01,28.75,445,104,0.0,down
1,content_4d1fe5b32dc2,client_19581e27de,2,0.934889,0.994167,0.8432,0.963733,0.866582,page_one_decay_risk|low_engagement_visible_page,monitor,...,512,549,2.5,0.52,7.47,13.15,329,104,0.0,stable
2,content_07f2e7a6f38a,client_19581e27de,3,0.934080,0.994467,0.8432,0.959965,0.866843,page_one_decay_risk|low_engagement_visible_page,monitor,...,856,780,2.7,0.85,2.05,4.60,313,104,0.0,stable
3,content_e5ae436f9a16,client_4e07408562,4,0.933606,0.996000,0.8432,0.955347,0.868180,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,533,522,3.0,0.45,7.09,12.60,421,104,0.0,stable
4,content_3430a8b94511,client_19581e27de,5,0.933559,0.998167,0.8432,0.951314,0.870069,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,440,534,3.3,0.29,6.18,11.04,329,104,0.0,stable


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I chose Logistic Regression because the target is binary and the model is interpretable. It provides a simple, explainable model for the Content Refresh Prioritization lane. I will compare it with the Week-4 baseline using the same client-holdout test set and evaluation metric.

In [213]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

print("Model:", model.__class__.__name__)
print("Purpose: binary classification for content refresh prioritization")

Model: LogisticRegression
Purpose: binary classification for content refresh prioritization


In [214]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I use a client-holdout split because pages from the same client may share patterns. A random row split could place pages from the same client in both train and test and make performance look artificially strong. GroupShuffleSplit keeps clients separated between the two sets.

In [215]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(
    "/content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv"
)

GROUP_COL = "client_id"

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df[GROUP_COL])
)

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print(f"Train: {len(train_df)} rows, {train_df[GROUP_COL].nunique()} clients")
print(f"Test: {len(test_df)} rows, {test_df[GROUP_COL].nunique()} clients")

assert set(train_df[GROUP_COL]).isdisjoint(
    set(test_df[GROUP_COL])
)

print("Client-holdout sanity check passed: no client appears in both sets.")

Train: 23837 rows, 25 clients
Test: 6163 rows, 7 clients
Client-holdout sanity check passed: no client appears in both sets.


In [216]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [217]:
print("results_df:", "results_df" in globals())
print("models:", "models" in globals())
print("X_test:", "X_test" in globals())
print("y_test:", "y_test" in globals())
print("FEATURE_COLS:", "FEATURE_COLS" in globals())

results_df: True
models: False
X_test: True
y_test: True
FEATURE_COLS: True


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Train + compare vs baseline

I train Logistic Regression to predict `is_declining_label` using the same client-holdout split as the Week-4 baseline. Because this lane produces a ranked queue, I compare the model and baseline using Precision@50 as the primary metric. Average Precision and ROC-AUC are included as supporting ranking metrics.

In [218]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# ============================================================
# 1. Target
# ============================================================

TARGET_COL = "is_declining_label"

y = df[TARGET_COL].astype(int)


# ============================================================
# 2. Build leakage-aware feature set
# ============================================================

exclude_cols = [
    "content_id",
    "client_id",

    # Target
    "is_declining_label",

    # Direct / near-direct target-derived signals
    "trend_direction",
    "trend_pct",
]

FEATURE_COLS = [
    col
    for col in df.select_dtypes(include=np.number).columns
    if col not in exclude_cols
]

X = (
    df[FEATURE_COLS]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("Number of features:", len(FEATURE_COLS))
print("Excluded:", [
    "trend_direction",
    "trend_pct"
])


# ============================================================
# 3. Same client-holdout split
# ============================================================

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


# ============================================================
# 4. Train Logistic Regression
# ============================================================

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

logistic_model.fit(X_train, y_train)

model_scores = logistic_model.predict_proba(X_test)[:, 1]

model_predictions = (
    model_scores >= 0.5
).astype(int)


# ============================================================
# 5. Precision@50
# ============================================================

def precision_at_k(y_true, scores, k=50):

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(-scores)[:k]

    return float(
        y_true[order].mean()
    )


# ============================================================
# 6. Logistic Regression metrics
# ============================================================

model_metrics = {
    "model": "logistic_regression",

    "accuracy": accuracy_score(
        y_test,
        model_predictions
    ),

    "precision": precision_score(
        y_test,
        model_predictions,
        zero_division=0
    ),

    "recall": recall_score(
        y_test,
        model_predictions,
        zero_division=0
    ),

    "f1": f1_score(
        y_test,
        model_predictions,
        zero_division=0
    ),

    "precision_at_50": precision_at_k(
        y_test,
        model_scores,
        50
    ),

    "average_precision": average_precision_score(
        y_test,
        model_scores
    ),

    "roc_auc": roc_auc_score(
        y_test,
        model_scores
    ),
}


# ============================================================
# 7. Week-4 baseline
# ============================================================

baseline_df = pd.read_csv(
    "/content/flyrank-ml-internship/"
    "data/processed/baseline_refresh_queue.csv"
)

baseline_lookup = baseline_df.set_index(
    "content_id"
)["baseline_refresh_score"]

test_content_ids = df.iloc[test_idx]["content_id"]

baseline_scores = (
    test_content_ids
    .map(baseline_lookup)
    .fillna(0)
    .to_numpy()
)


# ============================================================
# 8. Baseline metrics
# ============================================================

baseline_predictions = (
    baseline_scores >= 0.5
).astype(int)

baseline_metrics = {

    "model": "week4_baseline",

    "accuracy": accuracy_score(
        y_test,
        baseline_predictions
    ),

    "precision": precision_score(
        y_test,
        baseline_predictions,
        zero_division=0
    ),

    "recall": recall_score(
        y_test,
        baseline_predictions,
        zero_division=0
    ),

    "f1": f1_score(
        y_test,
        baseline_predictions,
        zero_division=0
    ),

    "precision_at_50": precision_at_k(
        y_test,
        baseline_scores,
        50
    ),

    "average_precision": average_precision_score(
        y_test,
        baseline_scores
    ),

    "roc_auc": roc_auc_score(
        y_test,
        baseline_scores
    ),
}


# ============================================================
# 9. Model vs baseline
# ============================================================

results_df = pd.DataFrame([
    baseline_metrics,
    model_metrics
])

results_df = (
    results_df
    .sort_values(
        "precision_at_50",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n=== LEAKAGE-AWARE MODEL VS WEEK-4 BASELINE ===")

display(
    results_df.round(4)
)

Number of features: 36
Excluded: ['trend_direction', 'trend_pct']
Training rows: 23837
Test rows: 6163

=== LEAKAGE-AWARE MODEL VS WEEK-4 BASELINE ===


,model,accuracy,precision,recall,f1,precision_at_50,average_precision,roc_auc
0,logistic_regression,0.7633,0.8086,0.7031,0.7522,1.00,0.8759,0.8584
1,week4_baseline,0.4676,0.4610,0.2477,0.3222,0.32,0.4819,0.4979


In [219]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

The Logistic Regression model achieved a measured Precision@50 of 1.00 on the held-out client test set, meaning all 50 highest-ranked pages were declining in this evaluation. The model made 524 false-positive and 935 false-negative predictions, so errors remain outside the highest-ranked portion of the queue.

Permutation importance shows that recent and historical search visibility were the strongest measured signals. `impressions_last_30d`, `impressions_prev_30d`, and `log_impressions_90d` had the largest importance values. This is directionally consistent with the Content Refresh Prioritization goal because declining visibility is useful decision-support signal for identifying pages to review.

I also removed `trend_direction` and `trend_pct` after discovering that they directly encoded the target. The reported model therefore excludes those target-derived fields. The results are measured on an unseen-client holdout and should be treated as decision-support evidence rather than proof that the model will perform identically on future data.dence rather than proof that the model will perform identically on future data.

In [220]:
# ============================================================
# SECTION 4 — Errors and interpretation
# ============================================================

from sklearn.inspection import permutation_importance

# ------------------------------------------------------------
# 1. Verify Precision@50
# ------------------------------------------------------------

top_50_idx = np.argsort(-model_scores)[:50]

top_50_actual = y_test.iloc[top_50_idx].to_numpy()

print("Top-50 predicted declining pages:")
print("Actual declining:", int(top_50_actual.sum()))
print("Top-50 Precision:", round(top_50_actual.mean(), 4))


# ------------------------------------------------------------
# 2. Permutation importance
# ------------------------------------------------------------

perm = permutation_importance(
    logistic_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="average_precision"
)

importance_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance": perm.importances_mean
}).sort_values(
    "importance",
    ascending=False
)

print("\n=== TOP 10 FEATURES BY PERMUTATION IMPORTANCE ===")

display(
    importance_df.head(10).round(4)
)


# ------------------------------------------------------------
# 3. Error breakdown
# ------------------------------------------------------------

error_df = df.iloc[test_idx].copy()

error_df["actual"] = y_test.to_numpy()
error_df["predicted"] = model_predictions
error_df["score"] = model_scores

error_df["error_type"] = "correct"

error_df.loc[
    (error_df["actual"] == 0) &
    (error_df["predicted"] == 1),
    "error_type"
] = "false_positive"

error_df.loc[
    (error_df["actual"] == 1) &
    (error_df["predicted"] == 0),
    "error_type"
] = "false_negative"

print("\n=== ERROR COUNTS ===")

print(
    error_df["error_type"]
    .value_counts()
)


# ------------------------------------------------------------
# 4. Error rates
# ------------------------------------------------------------

false_positives = (
    (y_test == 0) &
    (model_predictions == 1)
).sum()

false_negatives = (
    (y_test == 1) &
    (model_predictions == 0)
).sum()

correct = (
    y_test.to_numpy() == model_predictions
).sum()

print("\nCorrect predictions:", int(correct))
print("False positives:", int(false_positives))
print("False negatives:", int(false_negatives))


# ------------------------------------------------------------
# 5. Highest-confidence errors
# ------------------------------------------------------------

errors_only = error_df[
    error_df["error_type"] != "correct"
].copy()

errors_only["confidence"] = np.abs(
    errors_only["score"] - 0.5
)

print("\n=== HIGHEST-CONFIDENCE ERRORS ===")

display(
    errors_only
    .sort_values("confidence", ascending=False)
    [
        [
            "actual",
            "predicted",
            "score",
            "error_type"
        ]
    ]
    .head(10)
    .reset_index(drop=True)
)

Top-50 predicted declining pages:
Actual declining: 50
Top-50 Precision: 1.0

=== TOP 10 FEATURES BY PERMUTATION IMPORTANCE ===


,feature,importance
15,impressions_last_30d,0.3628
18,impressions_prev_30d,0.3069
29,log_impressions_90d,0.2140
13,days_with_impressions,0.0527
30,log_clicks_90d,0.0469
6,clicks_90d,0.0276
21,content_age_days,0.0126
25,avg_position,0.0121
16,clicks_last_30d,0.0119
19,clicks_prev_30d,0.0105



=== ERROR COUNTS ===
error_type
correct           4704
false_negative     935
false_positive     524
Name: count, dtype: int64

Correct predictions: 4704
False positives: 524
False negatives: 935

=== HIGHEST-CONFIDENCE ERRORS ===


,actual,predicted,score,error_type
0,0,1,0.927273,false_positive
1,1,0,0.086129,false_negative
2,0,1,0.911559,false_positive
3,1,0,0.090120,false_negative
4,0,1,0.907183,false_positive
5,1,0,0.096660,false_negative
6,1,0,0.097010,false_negative
7,1,0,0.098616,false_negative
8,1,0,0.103582,false_negative
9,1,0,0.108074,false_negative


In [221]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check

- [x] Every section is filled with both reasoning and supporting code.
- [x] The notebook uses a client-holdout split and verifies that no client appears in both train and test.
- [x] Logistic Regression is compared with the Week-4 baseline on the same held-out test set.
- [x] Precision@50 is used as the primary ranking metric, with Average Precision and ROC-AUC as supporting metrics.
- [x] `trend_direction` and `trend_pct` were excluded after the leakage diagnostic showed that `trend_direction` directly encoded the target.
- [x] Errors and permutation importance are reported.
- [x] Claims use careful language such as measured, observed, directional, and decision-support.
- [x] No client names, URLs, or private queries are included.
